# 01 — Data Loading, Cleaning & Merging

**Purpose**: Load IMDb TSV files and TMDb Kaggle dataset, clean, merge, and save unified dataframe.

**Outputs**:
- `data/movies_clean.parquet` — merged movie dataframe
- `data/actor_popularity.parquet` — actor popularity scores
- `data/director_stats.parquet` — director performance stats
- `data/valid_tconst_set.parquet` — valid movie IDs

**Key decisions** (see wiki Decision Log):
- D007: TMDb data from Kaggle alanvourch dataset (not API)
- D010: Min vote threshold configurable, default 500
- D012: UV as package manager
- D014: isAdult filter (stakeholder discussion point)

In [8]:
import pandas as pd
import numpy as np
import os
import zipfile
import warnings
import re
warnings.filterwarnings('ignore')

# Paths
SOURCE_DIR = os.path.join('..', 'source_data')
DATA_DIR = os.path.join('..', 'data')
os.makedirs(DATA_DIR, exist_ok=True)

# Config (D010: configurable min vote threshold)
MIN_VOTE_THRESHOLD = 500
CURRENT_YEAR = 2025

print('Config:')
print(f'  MIN_VOTE_THRESHOLD = {MIN_VOTE_THRESHOLD}')
print(f'  CURRENT_YEAR = {CURRENT_YEAR}')
print(f'  SOURCE_DIR = {os.path.abspath(SOURCE_DIR)}')
print(f'  DATA_DIR = {os.path.abspath(DATA_DIR)}')

Config:
  MIN_VOTE_THRESHOLD = 500
  CURRENT_YEAR = 2025
  SOURCE_DIR = C:\Users\Yael\Documents\SE_Projects\gcash_project\source_data
  DATA_DIR = C:\Users\Yael\Documents\SE_Projects\gcash_project\data


## 1. Load IMDb Datasets

Loading 5 TSV files from IMDb bulk data. Key notes:
- `\N` = null in IMDb format → replace with NaN
- Year columns (startYear, endYear) are float due to NaN → convert to nullable Int64
- Filter `titleType == 'movie'` (exclude shorts, TV, etc.)
- Filter `isAdult` to valid values only (0 or 1)

In [9]:
def load_imdb_tsv(filename, source_dir=SOURCE_DIR, usecols=None):
    """Load an IMDb TSV.gz file, replace \\N with NaN, fix year columns."""
    path = os.path.join(source_dir, filename)
    print(f'Loading {filename}...')
    df = pd.read_csv(path, sep='\t', compression='gzip', 
                     na_values='\\N', low_memory=False, usecols=usecols)
    
    # Fix year columns (float → nullable Int64)
    for col in ['startYear', 'endYear', 'birthYear', 'deathYear']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')
    
    print(f'  Shape: {df.shape}')
    print(f'  Columns: {list(df.columns)}')
    print(f'  Nulls:\n{df.isnull().sum()}')
    display(df.head(3))
    return df

In [10]:
# Load title.basics — movie metadata
title_basics = load_imdb_tsv('title.basics.tsv.gz')

# Print all titleType values (for inspection)
print('\n=== titleType values ===')
print(title_basics['titleType'].value_counts().to_string())

# Fix isAdult: filter to valid values only (0 or 1)
# NOTE: isAdult has corrupt data — year values (2023, 1979) mixed with booleans (0, 1)
print('\n=== isAdult values (before fix) ===')
print(title_basics['isAdult'].value_counts().head(20).to_string())

valid_adult = title_basics['isAdult'].isin([0, 1])
print(f'\nRows with valid isAdult: {valid_adult.sum():,} / {len(title_basics):,}')
title_basics = title_basics[valid_adult].copy()

# D014: Exclude adult content (3.3% of dataset)
# Stakeholder discussion: "We excluded adult content as it's not relevant for mainstream movie studio recommendations"
print('\n=== isAdult values (after fix) ===')
print(title_basics['isAdult'].value_counts().to_string())

Loading title.basics.tsv.gz...
  Shape: (12880486, 9)
  Columns: ['tconst', 'titleType', 'primaryTitle', 'originalTitle', 'isAdult', 'startYear', 'endYear', 'runtimeMinutes', 'genres']
  Nulls:
tconst                   0
titleType                0
primaryTitle            25
originalTitle           25
isAdult                  0
startYear          1486883
endYear           12717251
runtimeMinutes     8020170
genres              545389
dtype: int64


,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres
0,tt0000001,short,Carmencita,Carmencita,0,1894,<NA>,1,"Documentary,Short"
1,tt0000002,short,Le clown et ses chiens,Le clown et ses chiens,0,1892,<NA>,5,"Animation,Short"
2,tt0000003,short,Poor Pierrot,Pauvre Pierrot,0,1892,<NA>,5,"Animation,Comedy,Romance"



=== titleType values ===
titleType
tvEpisode       9962602
short           1164440
movie            759992
video            332589
tvSeries         307289
tvMovie          156638
tvMiniSeries      73718
tvSpecial         61419
videoGame         50716
tvShort           11082
tvPilot               1

=== isAdult values (before fix) ===
isAdult
0       12462970
1         417484
2023           7
1979           4
2019           3
2024           3
2025           3
2020           2
2022           2
2018           2
1981           1
2017           1
2014           1
2013           1
1978           1
2005           1

Rows with valid isAdult: 12,880,454 / 12,880,486

=== isAdult values (after fix) ===
isAdult
0    12462970
1      417484


In [11]:
# Filter to movies only
print(f'Before movie filter: {len(title_basics):,} rows')
title_basics = title_basics[title_basics['titleType'] == 'movie'].copy()
print(f'After movie filter: {len(title_basics):,} rows')
print(f'Start year range: {title_basics["startYear"].min()} - {title_basics["startYear"].max()}')
display(title_basics.head(3))

Before movie filter: 12,880,454 rows
After movie filter: 759,992 rows
Start year range: 1894 - 2032


,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres
8,tt0000009,movie,Miss Jerry,Miss Jerry,0,1894,<NA>,45,Romance
144,tt0000147,movie,The Corbett-Fitzsimmons Fight,The Corbett-Fitzsimmons Fight,0,1897,<NA>,100,"Documentary,News,Sport"
498,tt0000502,movie,Bohemios,Bohemios,0,1905,<NA>,100,NaN


In [12]:
# Load title.ratings — ratings and vote counts
title_ratings = load_imdb_tsv('title.ratings.tsv.gz')

# Filter by vote threshold (D010)
print(f'Before vote filter: {len(title_ratings):,} rows')
title_ratings = title_ratings[title_ratings['numVotes'] >= MIN_VOTE_THRESHOLD].copy()
print(f'After vote filter (>= {MIN_VOTE_THRESHOLD}): {len(title_ratings):,} rows')

Loading title.ratings.tsv.gz...
  Shape: (1722889, 3)
  Columns: ['tconst', 'averageRating', 'numVotes']
  Nulls:
tconst           0
averageRating    0
numVotes         0
dtype: int64


,tconst,averageRating,numVotes
0,tt0000001,5.7,2237
1,tt0000002,5.4,327
2,tt0000003,6.4,2398


Before vote filter: 1,722,889 rows
After vote filter (>= 500): 170,856 rows


In [13]:
# Compute valid tconst set = intersection of filtered title_basics and title_ratings
valid_tconsts = set(title_basics['tconst']) & set(title_ratings['tconst'])
print(f'Valid tconst set: {len(valid_tconsts):,} movies (all years, >= {MIN_VOTE_THRESHOLD} votes)')

# Save valid_tconst_set for EDA notebook
valid_tconst_df = pd.DataFrame({'tconst': list(valid_tconsts)})
valid_tconst_df.to_parquet(os.path.join(DATA_DIR, 'valid_tconst_set.parquet'), index=False)
print('Saved valid_tconst_set.parquet')

Valid tconst set: 71,056 movies (all years, >= 500 votes)
Saved valid_tconst_set.parquet


In [14]:
# Load title.principals in chunks (788MB file — too large to load entirely)
# Filter to valid tconsts AND actor/actress category
print('Loading title.principals.tsv.gz (chunked)...')
chunks = []
for i, chunk in enumerate(pd.read_csv(
    os.path.join(SOURCE_DIR, 'title.principals.tsv.gz'),
    sep='\t', compression='gzip', na_values='\\N',
    chunksize=1_000_000
)):
    chunk = chunk[
        (chunk['tconst'].isin(valid_tconsts)) & 
        (chunk['category'].isin(['actor', 'actress']))
    ]
    chunks.append(chunk)
    if (i + 1) % 10 == 0:
        print(f'  Processed {(i+1) * 1_000_000:,} rows...')

title_principals = pd.concat(chunks, ignore_index=True)
del chunks  # Free memory

print(f'\nFinal shape: {title_principals.shape}')
print(f'Unique actors: {title_principals["nconst"].nunique():,}')
print(f'Unique movies: {title_principals["tconst"].nunique():,}')
display(title_principals.head(3))

Loading title.principals.tsv.gz (chunked)...
  Processed 10,000,000 rows...
  Processed 20,000,000 rows...
  Processed 30,000,000 rows...
  Processed 40,000,000 rows...
  Processed 50,000,000 rows...
  Processed 60,000,000 rows...
  Processed 70,000,000 rows...
  Processed 80,000,000 rows...
  Processed 90,000,000 rows...
  Processed 100,000,000 rows...

Final shape: (672145, 6)
Unique actors: 250,278
Unique movies: 67,974


,tconst,ordering,nconst,category,job,characters
0,tt0000574,1,nm0846887,actress,NaN,"[""Kate Kelly""]"
1,tt0000574,2,nm0846894,actor,NaN,"[""School Master""]"
2,tt0000574,3,nm1431224,actor,NaN,"[""Joe Byrne""]"


In [16]:
# Load title.crew — director and writer IDs
# Filter to valid tconst set only
title_crew = load_imdb_tsv('title.crew.tsv.gz')
title_crew = title_crew[title_crew['tconst'].isin(valid_tconsts)].copy()
print(f'After filter to valid tconsts: {len(title_crew):,} rows')

Loading title.crew.tsv.gz...
  Shape: (12880486, 3)
  Columns: ['tconst', 'directors', 'writers']
  Nulls:
tconst             0
directors    5735860
writers      6375033
dtype: int64


,tconst,directors,writers
0,tt0000001,nm0005690,NaN
1,tt0000002,nm0721526,NaN
2,tt0000003,nm0721526,nm0721526


After filter to valid tconsts: 71,056 rows


In [17]:
# Load name.basics — person metadata
# Filter to only actors/actresses in our principals dataset
name_basics = load_imdb_tsv('name.basics.tsv.gz')

actor_nconsts = set(title_principals['nconst'])
print(f'Actor nconsts from principals: {len(actor_nconsts):,}')

name_basics = name_basics[name_basics['nconst'].isin(actor_nconsts)].copy()
print(f'After filter to relevant actors: {len(name_basics):,} rows')
display(name_basics.head(3))

Loading name.basics.tsv.gz...
  Shape: (15748361, 6)
  Columns: ['nconst', 'primaryName', 'birthYear', 'deathYear', 'primaryProfession', 'knownForTitles']
  Nulls:
nconst                      0
primaryName               100
birthYear            15064232
deathYear            15483740
primaryProfession     3198554
knownForTitles        1906181
dtype: int64


,nconst,primaryName,birthYear,deathYear,primaryProfession,knownForTitles
0,nm0000001,Fred Astaire,1899,1987,"actor,miscellaneous,producer","tt0072308,tt0031983,tt0050419,tt0025164"
1,nm0000002,Lauren Bacall,1924,2014,"actress,miscellaneous,soundtrack","tt0037382,tt0117057,tt0075213,tt0038355"
2,nm0000003,Brigitte Bardot,1934,2025,"actress,music_department,producer","tt0057345,tt0049189,tt0056404,tt0054452"


Actor nconsts from principals: 250,278
After filter to relevant actors: 250,266 rows


,nconst,primaryName,birthYear,deathYear,primaryProfession,knownForTitles
0,nm0000001,Fred Astaire,1899,1987,"actor,miscellaneous,producer","tt0072308,tt0031983,tt0050419,tt0025164"
1,nm0000002,Lauren Bacall,1924,2014,"actress,miscellaneous,soundtrack","tt0037382,tt0117057,tt0075213,tt0038355"
2,nm0000003,Brigitte Bardot,1934,2025,"actress,music_department,producer","tt0057345,tt0049189,tt0056404,tt0054452"


## 2. Load TMDb Kaggle Dataset (D007)

alanvourch's "Ultimate 1M Movies Dataset (TMDB + IMDb)" from Kaggle.
- Has `imdb_id` for joining with IMDb data
- Provides budget, revenue, cast, director, genres
- Filtered to valid tconst set to reduce memory

In [18]:
# Load TMDb CSV from zip
tmdb_zip = os.path.join(SOURCE_DIR, 'alanvourch-tmdb-movies-daily-updates.zip')
print(f'Loading TMDb from {tmdb_zip}...')

with zipfile.ZipFile(tmdb_zip) as z:
    with z.open('TMDB_all_movies.csv') as f:
        tmdb = pd.read_csv(f, low_memory=False)

# Rename and filter
tmdb = tmdb.rename(columns={'imdb_id': 'tconst'})
tmdb_cols = ['tconst', 'budget', 'revenue', 'popularity', 'production_companies', 
             'production_countries', 'certification_us', 'keywords']
tmdb = tmdb[tmdb_cols].drop_duplicates(subset='tconst', keep='first')
tmdb = tmdb[tmdb['tconst'].isin(valid_tconsts)].copy()
print(f'TMDb after filter: {len(tmdb):,} rows')

# Free memory
del valid_tconst_df
print('Freed: valid_tconst_df')

Loading TMDb from ..\source_data\alanvourch-tmdb-movies-daily-updates.zip...
TMDb after filter: 70,337 rows
Freed: valid_tconst_df


## 3. Merge Datasets

Merge order:
1. `title_basics` + `title_ratings` → `movies`
2. `movies` + `title_crew` → add director/writer IDs
3. `movies` + `name_basics` → add director names
4. `movies` + `tmdb` → add budget, revenue, etc.

In [19]:
# Step 1: Merge basics + ratings
movies = title_basics.merge(title_ratings, on='tconst', how='inner')
print(f'After basics + ratings merge: {len(movies):,} rows')

# Step 2: Merge with crew (directors)
movies = movies.merge(title_crew, on='tconst', how='left')
movies['primary_director_nconst'] = movies['directors'].str.split(',').str[0]

# Step 3: Merge with name_basics to get director name
director_names = name_basics[['nconst', 'primaryName']].rename(
    columns={'nconst': 'primary_director_nconst', 'primaryName': 'director_name'}
)
movies = movies.merge(director_names, on='primary_director_nconst', how='left')
print(f'After crew merge: {len(movies):,} rows')
print(f'Has director name: {movies["director_name"].notna().sum():,}')

# Step 4: Merge with TMDb
movies = movies.merge(tmdb, on='tconst', how='left')
print(f'After TMDb merge: {len(movies):,} rows')
print(f'Has budget > 0: {(movies["budget"] > 0).sum():,}')
print(f'Has revenue > 0: {(movies["revenue"] > 0).sum():,}')

# Free memory: title_basics, title_crew, tmdb, director_names
del title_basics, title_crew, tmdb, director_names
print('Freed: title_basics, title_crew, tmdb, director_names')

display(movies[['tconst', 'primaryTitle', 'startYear', 'averageRating', 'numVotes', 'director_name', 'budget', 'revenue']].head(3))

After basics + ratings merge: 71,056 rows
After crew merge: 71,056 rows
Has director name: 16,180
After TMDb merge: 71,056 rows
Has budget > 0: 16,968
Has revenue > 0: 18,445
Freed: title_basics, title_crew, tmdb, director_names


,tconst,primaryTitle,startYear,averageRating,numVotes,director_name,budget,revenue
0,tt0000147,The Corbett-Fitzsimmons Fight,1897,5.2,616,NaN,0.0,0.0
1,tt0000574,The Story of the Kelly Gang,1906,6.0,1102,NaN,0.0,0.0
2,tt0002101,Cleopatra,1912,5.1,676,NaN,0.0,0.0


## 4. Compute Actor Popularity (D008, D009)

Formula: `Actor Popularity = SUM( (1/log(ordering+1)) × numVotes × averageRating )`

Computed across ALL movies (historical data) — not just 2023.

In [20]:
# Merge principals with ratings
principals_with_ratings = title_principals.merge(title_ratings, on='tconst', how='left')

# Compute billing weight (D009: 1/log(ordering+1))
principals_with_ratings['billing_weight'] = 1.0 / np.log(
    principals_with_ratings['ordering'].astype(float) + 1
)

# Compute weighted score per appearance
principals_with_ratings['weighted_score'] = (
    principals_with_ratings['billing_weight'] * 
    principals_with_ratings['numVotes'].fillna(0) * 
    principals_with_ratings['averageRating'].fillna(0)
)

# Aggregate by actor
actor_popularity = principals_with_ratings.groupby('nconst').agg(
    total_score=('weighted_score', 'sum'),
    num_movies=('tconst', 'nunique'),
    total_votes=('numVotes', 'sum')
).reset_index()

# Add actor names
actor_names = name_basics[['nconst', 'primaryName']].drop_duplicates()
actor_popularity = actor_popularity.merge(actor_names, on='nconst', how='left')

# Sort by popularity
actor_popularity = actor_popularity.sort_values('total_score', ascending=False)

print(f'Top 20 Most Popular Actors (across all years in dataset):')
display(actor_popularity.head(20)[['primaryName', 'total_score', 'num_movies', 'total_votes']])

# Free memory: title_principals, title_ratings, name_basics, principals_with_ratings, actor_names
del title_principals, title_ratings, name_basics, principals_with_ratings, actor_names
print('Freed: title_principals, title_ratings, name_basics, principals_with_ratings, actor_names')

Top 20 Most Popular Actors (across all years in dataset):


,primaryName,total_score,num_movies,total_votes
119,Leonardo DiCaprio,2.214441e+08,34,20477945
139,Tom Hanks,2.149897e+08,63,20561457
77,Brad Pitt,2.034374e+08,54,22403566
343,Robert Downey Jr.,1.993165e+08,59,23781311
262,Christian Bale,1.879829e+08,47,18266830
115,Robert De Niro,1.567598e+08,100,18313469
110,Tom Cruise,1.405013e+08,45,13918094
323,Matt Damon,1.401150e+08,56,16804864
30839,Chris Evans,1.356115e+08,36,21320258
46462,Scarlett Johansson,1.348150e+08,55,27272374


Freed: title_principals, title_ratings, name_basics, principals_with_ratings, actor_names


## 5. Compute Director Performance

In [21]:
# Director stats from filtered movies
director_stats = movies.groupby('director_name').agg(
    num_movies=('tconst', 'nunique'),
    avg_rating=('averageRating', 'mean'),
    avg_votes=('numVotes', 'mean'),
    total_votes=('numVotes', 'sum')
).reset_index()

director_stats = director_stats.sort_values('total_votes', ascending=False)

print(f'Top 20 Directors by Total Audience Reach:')
display(director_stats.head(20))

Top 20 Directors by Total Audience Reach:


,director_name,num_movies,avg_rating,avg_votes,total_votes
3778,Quentin Tarantino,12,7.850000,1.012903e+06,12154836
3024,Martin Scorsese,36,7.494444,3.324221e+05,11967195
3670,Peter Jackson,15,7.686667,6.561999e+05,9842999
945,Clint Eastwood,40,6.947500,1.533589e+05,6134355
4049,Ron Howard,34,6.847059,1.573206e+05,5348901
4151,Sam Raimi,16,6.818750,2.998738e+05,4797981
2869,M. Night Shyamalan,16,6.012500,2.961171e+05,4737873
1630,George Lucas,6,7.233333,7.452098e+05,4471259
197,Alfred Hitchcock,53,6.873585,8.134802e+04,4311445
4461,Steven Soderbergh,38,6.565789,1.047231e+05,3979476


## 6. Save All Dataframes

In [22]:
# Save all dataframes
movies.to_parquet(os.path.join(DATA_DIR, 'movies_clean.parquet'), index=False)
actor_popularity.to_parquet(os.path.join(DATA_DIR, 'actor_popularity.parquet'), index=False)
director_stats.to_parquet(os.path.join(DATA_DIR, 'director_stats.parquet'), index=False)

print('Saved:')
print(f'  movies_clean.parquet ({os.path.getsize(os.path.join(DATA_DIR, "movies_clean.parquet")) / 1024 / 1024:.1f} MB)')
print(f'  actor_popularity.parquet ({os.path.getsize(os.path.join(DATA_DIR, "actor_popularity.parquet")) / 1024 / 1024:.1f} MB)')
print(f'  director_stats.parquet ({os.path.getsize(os.path.join(DATA_DIR, "director_stats.parquet")) / 1024 / 1024:.1f} MB)')

Saved:
  movies_clean.parquet (9.7 MB)
  actor_popularity.parquet (8.1 MB)
  director_stats.parquet (0.1 MB)


## 7. Quick 2023 Snapshot

Preview 2023 movies for EDA Q1 and Q2.

In [23]:
# 2023 movies (filter from already-loaded data)
movies_2023 = movies[movies['startYear'] == 2023].copy()
print(f'2023 movies with >= {MIN_VOTE_THRESHOLD} votes: {len(movies_2023):,}')

# Top 10 by rating
print(f'\nTop 10 Highest Rated 2023 Movies:')
display(movies_2023.nlargest(10, 'averageRating')[
    ['primaryTitle', 'averageRating', 'numVotes', 'director_name']
])

# Top 10 by votes (most popular)
print(f'\nTop 10 Most Voted 2023 Movies:')
display(movies_2023.nlargest(10, 'numVotes')[
    ['primaryTitle', 'averageRating', 'numVotes', 'director_name']
])

2023 movies with >= 500 votes: 2,529

Top 10 Highest Rated 2023 Movies:


,primaryTitle,averageRating,numVotes,director_name
60486,I Forced an AI to Play a Kids Adventure Game,9.4,546,NaN
33345,Mindgame,9.2,757,NaN
48061,Long Drive,9.2,834,NaN
52601,Suraari,9.2,774,Vishalajay
53349,Nimmellara Aashirvada,9.2,621,NaN
53090,90 Bidi Manig Nadi,9.1,533,NaN
53168,Detroit: Become Human,9.1,726,NaN
44450,Divorce,9.0,622,Mini I G
52099,Beega,9.0,735,H.M. Srinandan
53694,THEVR10: A dokumentumfilm,8.9,1270,NaN



Top 10 Most Voted 2023 Movies:


,primaryTitle,averageRating,numVotes,director_name
42533,Oppenheimer,8.2,1106550,NaN
42060,Barbie,6.8,708753,Greta Gerwig
70422,Spider-Man: Across the Spider-Verse,8.5,558940,NaN
66377,Guardians of the Galaxy Vol. 3,7.9,491185,James Gunn
40601,Poor Things,7.7,425924,Yorgos Lanthimos
33853,John Wick: Chapter 4,7.6,421497,NaN
70657,Mission: Impossible - Dead Reckoning Part One,7.6,334247,NaN
63705,Killers of the Flower Moon,7.5,315287,Martin Scorsese
66249,The Super Mario Bros. Movie,7.0,303637,NaN
35069,Ant-Man and the Wasp: Quantumania,6.0,281210,NaN


In [24]:
# Final summary
print('='*60)
print('FINAL DATAFRAME SUMMARY')
print('='*60)
print(f'Shape: {movies.shape}')
print(f'\nColumns: {list(movies.columns)}')
print(f'\nDtypes:\n{movies.dtypes}')
print(f'\nNull counts:\n{movies.isnull().sum()}')
print(f'\nSample (top rated):')
display(movies.nlargest(5, 'averageRating')[
    ['tconst', 'primaryTitle', 'startYear', 'averageRating', 'numVotes', 'director_name', 'budget', 'revenue']
])

FINAL DATAFRAME SUMMARY
Shape: (71056, 22)

Columns: ['tconst', 'titleType', 'primaryTitle', 'originalTitle', 'isAdult', 'startYear', 'endYear', 'runtimeMinutes', 'genres', 'averageRating', 'numVotes', 'directors', 'writers', 'primary_director_nconst', 'director_name', 'budget', 'revenue', 'popularity', 'production_companies', 'production_countries', 'certification_us', 'keywords']

Dtypes:
tconst                         str
titleType                      str
primaryTitle                   str
originalTitle                  str
isAdult                      int64
startYear                    Int64
endYear                      Int64
runtimeMinutes                 str
genres                         str
averageRating              float64
numVotes                     int64
directors                      str
writers                        str
primary_director_nconst     object
director_name                  str
budget                     float64
revenue                    float64
popularity 

,tconst,primaryTitle,startYear,averageRating,numVotes,director_name,budget,revenue
59502,tt38608685,Sohla: Celebration Under the Grey Sky,2026,10.0,1232,NaN,0.0,0.0
59878,tt39378361,Mayookha,2026,10.0,2070,Shree Tej SK,278000.0,0.0
40769,tt1433175,Romantic Weirdos and the Land of Oddz,2007,9.9,625,NaN,0.0,0.0
58659,tt36853476,Om Ka Hari,2026,9.9,5023,NaN,0.0,0.0
59738,tt39089210,All Pass,2026,9.9,1084,NaN,0.0,0.0
